# 🌍 Тестирование NASA Weather Data API

Этот notebook проверяет подключение к NASA POWER API и исследует структуру данных

In [ ]:
# Импорт библиотек
import sys
sys.path.append('..')

from weather_analysis import WeatherDataService
from weather_analysis import analyze_weather
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Тестирование подключения к API

In [ ]:
# Создаем сервис
service = WeatherDataService(preferred_source='nasa')

# Проверяем доступность источников
print("🔍 Проверка доступности источников данных...\n")
sources = service.test_connection()

for source, available in sources.items():
    status = "✅ Доступен" if available else "❌ Недоступен"
    print(f"{source}: {status}")

## 2. Получение данных для Москвы

In [ ]:
# Координаты Москвы
lat, lon = 55.7558, 37.6173

# Получаем данные за последние 10 лет
data, source = service.get_data(lat, lon, start_year=2014, end_year=2023)

print(f"\n✓ Источник: {source}")
print(f"✓ Размер данных: {data.shape}")
print(f"\nПервые 5 строк:")
data.head()

## 3. Исследование структуры данных

In [ ]:
# Информация о датафрейме
print("📊 Информация о данных:\n")
data.info()

print("\n📈 Статистика:\n")
data.describe()

## 4. Визуализация температуры по дням года

In [ ]:
# График средней температуры
plt.figure(figsize=(15, 6))

plt.plot(data['day_of_year'], data['T2M'], label='Средняя температура', linewidth=2)
plt.plot(data['day_of_year'], data['T2M_MAX'], label='Максимальная температура', alpha=0.7, linestyle='--')
plt.plot(data['day_of_year'], data['T2M_MIN'], label='Минимальная температура', alpha=0.7, linestyle='--')

plt.xlabel('День года', fontsize=12)
plt.ylabel('Температура (°C)', fontsize=12)
plt.title('🌡️ Климатология температуры в Москве (2014-2023)', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Визуализация осадков и ветра

In [ ]:
# Создаем два графика
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# График осадков
ax1.plot(data['day_of_year'], data['PRECTOTCORR'], color='blue', linewidth=2)
ax1.set_xlabel('День года', fontsize=12)
ax1.set_ylabel('Осадки (мм/день)', fontsize=12)
ax1.set_title('💧 Климатология осадков в Москве', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# График ветра
ax2.plot(data['day_of_year'], data['WS2M'], color='green', linewidth=2)
ax2.set_xlabel('День года', fontsize=12)
ax2.set_ylabel('Скорость ветра (м/с)', fontsize=12)
ax2.set_title('💨 Климатология ветра в Москве', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Тестирование функции analyze_weather

In [ ]:
# Анализ для конкретной даты
result = analyze_weather(
    latitude=55.7558,
    longitude=37.6173,
    date='2024-07-15'  # Середина лета
)

# Результат уже напечатан функцией

In [ ]:
# Можно также работать с результатом программно
probs = result['probabilities']

# Визуализируем вероятности
categories = ['Очень\nхолодно', 'Холодно', 'Комфортно', 'Жарко', 'Очень\nжарко', 
              'Очень\nвлажно', 'Очень\nветрено', 'Очень\nнекомфортно']
values = [
    probs.get('very_cold', 0),
    probs.get('cold', 0),
    probs.get('comfortable', 0),
    probs.get('hot', 0),
    probs.get('very_hot', 0),
    probs.get('very_wet', 0),
    probs.get('very_windy', 0),
    probs.get('very_uncomfortable', 0)
]

plt.figure(figsize=(12, 6))
bars = plt.bar(categories, [v*100 for v in values], color='skyblue', edgecolor='navy', linewidth=2)

# Раскрашиваем в зависимости от значения
for i, bar in enumerate(bars):
    if values[i] > 0.5:
        bar.set_color('red')
    elif values[i] > 0.25:
        bar.set_color('orange')

plt.ylabel('Вероятность (%)', fontsize=12)
plt.title(f"📊 Вероятности погодных условий - {result['date_name']}", fontsize=14, fontweight='bold')
plt.xticks(rotation=0, fontsize=10)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## ✅ Заключение

API работает! Данные успешно получены и проанализированы. Можно использовать этот модуль для работы.